In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent

src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))
    
from config import *
print(f"Environment linked. Data dir: {DATA_DIR}")

In [ ]:
import glob
import h5py
import numpy as np
from tqdm.notebook import tqdm


# We trim the first 20,000 samples to ensure the 'Jamming' class 
# contains actual interference, not the 'warm-up' silence.
START_OFFSET = 20000

In [ ]:
def convert_mat_to_npy(file_path, save_at_folder_path):
    """
    Reads a JamShield .mat file, extracts 3 scenarios, trims the start,
    and saves them as separate .npy files.
    """
    path_obj = Path(file_path)
    filename = path_obj.name
    base_name = path_obj.stem # e.g. 'w1'
    
    # Mapping: Internal Key -> Output Suffix
    scenarios = {
        'Nojamming': 'clean',
        'Sine':      'sine',
        'Gaussian':  'gaussian'
    }
    
    saved_count = 0
    
    try:
        with h5py.File(file_path, 'r') as f:
            for key, suffix in scenarios.items():
                if key in f:
                    dataset = f[key]
                    
                    # Safety Check: Is file long enough?
                    if dataset.shape[1] <= START_OFFSET:
                        continue
                        
                    # Slice: From Offset to End
                    # Shape is (2, N)
                    raw_slice = dataset[:, START_OFFSET:] 
                    
                    # Reconstruct Complex
                    complex_data = raw_slice[0, :] + 1j * raw_slice[1, :]
                    
                    # Save
                    output_name = f"{base_name}_{suffix}.npy"
                    np.save(save_at_folder_path / output_name, complex_data)
                    saved_count += 1

    except Exception as e:
        print(f"Error reading {filename}: {e}")
            
    return saved_count

In [ ]:
# Find Files
search_pattern = str(JAMSHIELD_RAW_DIR / JAMSHIELD_FILE_PATTERN)
files = sorted(glob.glob(search_pattern))

print(f"Found {len(files)} source files. Starting conversion...")
print("="*60)

# Run Batch
total_converted = 0
for f in tqdm(files, desc="Converting"):
    count = convert_mat_to_npy(f, JAMSHIELD_NPY_DIR)
    total_converted += count

print("="*60)
print(f"Processing Complete.")
print(f"Total .npy files created: {total_converted}")

In [ ]:
# Check output
npy_files = sorted(list(JAMSHIELD_NPY_DIR.glob("*.npy")))

if npy_files:
    print(f"Verification - Found {len(npy_files)} files.")
    test_file = npy_files[0]
    data = np.load(test_file)
    
    print(f"\nInspecting: {test_file.name}")
    print(f"Shape: {data.shape}")
    print(f"Dtype: {data.dtype}")
    
    if np.any(np.isnan(data)):
        print("WARNING: NaNs detected!")
    else:
        print("Integrity Check Passed.")
else:
    print("ERROR: No output files found!")